In [1]:
"""
%pip install qiskit-nature==0.7.2
%pip install qiskit-aer==0.17.2
%pip install qiskit-ibm-runtime==0.47.0
%pip install mapomatic==0.14.0
%pip install numpy==2.4.2
%pip install pyscf==2.12.1
%pip install openpyxl==3.1.5
"""

'\n%pip install qiskit-nature==0.7.2\n%pip install qiskit-aer==0.17.2\n%pip install qiskit-ibm-runtime==0.47.0\n%pip install mapomatic==0.14.0\n%pip install numpy==2.4.2\n%pip install pyscf==2.12.1\n%pip install openpyxl==3.1.5\n'

In [2]:
# Data analysis and visualization imports

import numpy as np
import pandas as pd
import networkx as nx
from statistics import median
import matplotlib.pyplot as plt

# Multiprocessing imports
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed

# IBM Runtime specific imports
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import EstimatorV2
from qiskit_ibm_runtime.fake_provider import (FakeMarrakesh,
                                              FakeFez,
                                              FakeKingston,
                                              FakeBoston,
                                              FakePittsburgh,
                                              FakeMiami,
                                              FakeAachen,
                                              FakeBerlin
                                              )
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Qiskit Nature and Algorithms specific imports
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_algorithms import MinimumEigensolverResult
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer

In [3]:
import sys

import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Python:", sys.version.split()[0])
print("qiskit:", qiskit.__version__)
print("qiskit-aer:", qiskit_aer.__version__)
print("qiskit-ibm-runtime:", qiskit_ibm_runtime.__version__)

Python: 3.12.13
qiskit: 2.5.1
qiskit-aer: 0.17.2
qiskit-ibm-runtime: 0.47.0


# 1°: Configuring fake providers.

In [4]:
fake_marrakesh = FakeMarrakesh()
fake_fez = FakeFez()
fake_kingston = FakeKingston()
fake_boston = FakeBoston()
fake_pittsburgh = FakePittsburgh()
fake_miami = FakeMiami()
fake_aachen = FakeAachen()
fake_berlin = FakeBerlin()

# 2°: Functions for analyzing qubit metadata and overall instance performance.

In [5]:
def get_backend_properties_df(backend):
  """
  Extracts properties of a given Qiskit backend and returns them as a pandas DataFrame.

  Args:
    backend: The Qiskit backend object (e.g., from QiskitRuntimeService.least_busy).

  Returns:
    pd.DataFrame: A single-row DataFrame containing the backend's metrics.
  """
  t1, t2 = [], []
  one_qubit_error = []
  readout_error = []

  qubits = backend.num_qubits
  properties = backend.properties()

  tgt = backend.target
  gates = set(tgt.keys())

  if backend.coupling_map:
      edges = list(backend.coupling_map.get_edges())
      qubit_pairs = {tuple(f) for f in map(frozenset, edges)}
  else:
      qubit_pairs = set()

  for qubit in range(qubits):
    qp = properties.qubit_property(qubit)
    if 'T1' in qp:
        t1.append(qp['T1'][0])
    if 'T2' in qp:
        t2.append(qp['T2'][0])
    if 'readout_error' in qp:
        readout_error.append(qp['readout_error'][0])

    ip1q = tgt.get('sx', {}).get((qubit,))
    if ip1q and getattr(ip1q, 'error', None) is not None:
      one_qubit_error.append(ip1q.error)

  relevant_2q_gates = ['cz', 'rzz', 'ecr', 'cx']
  available_2q_gates = [g for g in relevant_2q_gates if g in gates]

  medians_2q = {}

  for gname in available_2q_gates:
      tgt2q = tgt[gname]
      errors = []
      for a, b in qubit_pairs:
          ip = tgt2q.get((a, b)) or tgt2q.get((b, a))
          if ip and getattr(ip, 'error', None) is not None:
              errors.append(ip.error)
      medians_2q[f"{gname.upper()} median error"] = median(errors) if errors else None

  backend_name = backend.backend_name
  processor_type = "N/A"
  if backend.configuration() and 'processor_type' in backend.configuration():
      proc = backend.configuration().processor_type
      processor_type = f"{proc['family']}{proc['revision']}"

  dtm = properties.last_update_date
  sdt = dtm.strftime("%d/%m/%Y %H:%M")

  data = {
      "Backend Name": [backend_name],
      "Processor": [processor_type],
      "Snapshot Date": [sdt],
      "Qubits": [qubits],
      "T1 median [µs]": [median(t1) * 1e6 if t1 else None],
      "T2 median [µs]": [median(t2) * 1e6 if t2 else None],
      "Readout error median": [median(readout_error) if readout_error else None],
      "SX median error": [median(one_qubit_error) if one_qubit_error else None],
  }

  for col_name, med_val in medians_2q.items():
      data[col_name] = [med_val]

  return pd.DataFrame(data)

# 3°: General analysis of fake providers.

In [6]:
fake_providers = [fake_marrakesh,
                  fake_fez,
                  fake_kingston,
                  fake_boston,
                  fake_pittsburgh,
                  fake_miami,
                  fake_aachen,
                  fake_berlin
              ]
df_list = []

for fake_provider in fake_providers:
  df = get_backend_properties_df(fake_provider)
  df_list.append(df)

fake_provider_df = pd.concat(df_list, ignore_index=True)

fake_provider_df = fake_provider_df.sort_values(by="Backend Name", ascending=True).style.format({
    "T1 median [µs]": "{:.2f}",
    "T2 median [µs]": "{:.2f}",
    "Readout error median": "{:.2e}",
    "SX median error": "{:.2e}",
    "CZ median error": "{:.2e}",
    "RZZ median error": "{:.2e}",
    "ECR median error": "{:.2e}",
    "CX median error": "{:.2e}"
})

display(fake_provider_df)

,Backend Name,Processor,Snapshot Date,Qubits,T1 median [µs],T2 median [µs],Readout error median,SX median error,CZ median error
6,fake_aachen,Heron3,17/04/2026 12:29,156,232.35,255.43,7.39e-03,1.91e-04,1.54e-03
7,fake_berlin,Nighthawk1,17/04/2026 07:56,120,323.82,248.44,2.08e-02,2.11e-04,2.19e-03
3,fake_boston,Heron3,17/04/2026 12:19,156,292.44,352.97,5.13e-03,1.45e-04,1.27e-03
1,fake_fez,Heron2,26/02/2025 15:16,156,144.86,87.95,7.57e-03,2.29e-04,3.90e-03
2,fake_kingston,Heron2,15/04/2026 09:15,156,282.82,144.23,9.52e-03,2.36e-04,1.82e-03
0,fake_marrakesh,Heron2,26/02/2025 14:52,156,197.36,118.43,9.52e-03,2.30e-04,3.30e-03
5,fake_miami,Nighthawk1,17/04/2026 10:46,120,330.22,241.91,1.79e-02,1.97e-04,2.98e-03
4,fake_pittsburgh,Heron3,17/04/2026 11:46,156,298.97,317.10,4.52e-03,1.99e-04,1.54e-03


In [7]:
fake_provider_df.to_excel('fake_provider_df.xlsx', index=False)

# 4°: Setting up the chemical problem for the simulations and QPU selection.

In [8]:
# Creating the molecular geometry of BeH2 with STO-3G as minimal basis set and
# H-Be distance of 1.326 angstrom and 180 degrees.

driver = PySCFDriver(
        atom="""H -1.326, 0.0, 0.0
                Be 0.0, 0.0 0.0
                H 1.326, 0.0, 0.0
             """,
        basis='sto3g',
        charge=0,
        spin=0,
        unit=DistanceUnit.ANGSTROM
)

molecule_problem = driver.run()

In [9]:
# The original space has approximately 6 electrons, 7 space orbitals, and 14
# spin orbitals.
# Here we limit the active space to only 4 electrons and 3 space orbitals and
# work with the reduced molecule_problem

active_space_transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=3)
reduced_molecule_problem = active_space_transformer.transform(molecule_problem)

In [10]:
# Here we calculate the Hamiltonian of the second quantization after reduction
# with CAS.
second_q_hamiltonian = reduced_molecule_problem.second_q_ops()[0]

In [11]:
# Here we use the Jordan Wigner mapping
jordan_wigner_mapper = JordanWignerMapper()
qubit_op = jordan_wigner_mapper.map(second_q_hamiltonian)

## 4.1: Ansatz Circuit Construction

In [12]:
# Here we construct the HF state within the CAS and JW mapping.
hf_initial_state = HartreeFock(
      num_particles=reduced_molecule_problem.num_particles,
      num_spatial_orbitals=reduced_molecule_problem.num_spatial_orbitals,
      qubit_mapper=jordan_wigner_mapper
)

In [13]:
# Here we build the ansatz
ansatz = UCCSD(
          reduced_molecule_problem.num_spatial_orbitals,
          reduced_molecule_problem.num_particles,
          initial_state=hf_initial_state,
          qubit_mapper=jordan_wigner_mapper
)

## 4.2 Transpilation and eingenvalue functions

In [14]:
# This small function was created to generate the ansatz and observables in
# terms of Instruction Set Architecture (ISA) operators.
def transpile_to_isa(backend,  optimization_level=0, seed=42, initial_layout=None):
  target = backend.target

  pm = generate_preset_pass_manager(
       target=target,
       initial_layout=initial_layout,
       layout_method='sabre',
       routing_method='sabre',
       optimization_level=optimization_level,
       seed_transpiler=seed
  )

  ansatz_isa = pm.run(ansatz)
  isa_observables = qubit_op.apply_layout(ansatz_isa.layout)
  return ansatz_isa, isa_observables

In [15]:
# Here we add the energy of active space to the frozen energies of the core
# plus nuclear repulsion.
def interpret_exp_val(exp_val, problem):
    sol = MinimumEigensolverResult()
    sol.eigenvalue = np.real(exp_val)
    return problem.interpret(sol).total_energies[0]

# 5° Multiple transpilations for best seed detection

In [16]:
def evaluate_single_seed(seed, ansatz_circuit, target, opt_level):
    """
    Worker function to transpile the circuit with a specific seed.
    Returns lightweight metadata to avoid multiprocessing memory bottlenecks.
    """
    pm = generate_preset_pass_manager(
        target=target,
        layout_method='sabre',
        routing_method='sabre',
        optimization_level=opt_level,
        seed_transpiler=seed
    )
    transpiled_qc = pm.run(ansatz_circuit)

    # Extract operations count
    gate_counts = transpiled_qc.count_ops()

    # Safely count all possible native two-qubit gates across architectures.
    # Heron uses 'cz' and natively supports 'rzz'.
    # Nighthawk relies entirely on 'ecr' or 'cx' depending on the revision.
    two_q_count = (
        gate_counts.get('cz', 0) +
        gate_counts.get('rzz', 0) +
        gate_counts.get('ecr', 0) +
        gate_counts.get('cx', 0)
    )

    # Extract the physical qubit mapping (initial layout)
    # Maps the original virtual qubits to their chosen physical counterparts
    layout = transpiled_qc.layout.initial_layout
    if layout is not None:
        physical_qubits = [layout[v] for v in ansatz_circuit.qubits]
    else:
        # Fallback in case no layout was applied (rare at opt_level=3)
        physical_qubits = list(range(ansatz_circuit.num_qubits))

    return {
        'seed': seed,
        'depth': transpiled_qc.depth(),
        'two_q_count': two_q_count,
        'best_qubits': physical_qubits
    }

In [17]:
def find_best_seed_parallel(ansatz_circuit, target, num_seeds=200, opt_level=3):
    """
    Manages the parallel execution of multiple seeds for a single target backend.
    """
    best_result = {
        'seed': None,
        'two_q_count': float('inf'),
        'depth': float('inf'),
        'best_qubits': []
    }

    seeds_to_test = list(range(num_seeds))
    num_cores = multiprocessing.cpu_count()

    with ProcessPoolExecutor(max_workers=num_cores) as executor:
        # Submit all seed evaluations to the process pool
        futures = {
            executor.submit(evaluate_single_seed, seed, ansatz_circuit, target, opt_level): seed
            for seed in seeds_to_test
        }

        for future in as_completed(futures):
            res = future.result()

            # Decision criteria:
            # 1st Priority: Lowest number of 2-qubit gates (minimizing SWAP overhead).
            # 2nd Priority: Tie-breaker is the lowest overall depth.
            if (res['two_q_count'] < best_result['two_q_count']) or \
               (res['two_q_count'] == best_result['two_q_count'] and res['depth'] < best_result['depth']):
                best_result = res

    return best_result

In [18]:
benchmark_results = []
num_seeds_to_test = 500

print(f"Starting parallel seed search ({num_seeds_to_test} seeds per backend).")
print(f"Using {multiprocessing.cpu_count()} CPU cores.\n")

for provider in fake_providers:
    backend_name = provider.backend_name
    print(f"[{backend_name}] Scanning seeds...")

    # Run the parallel search
    best_metrics = find_best_seed_parallel(
        ansatz_circuit=ansatz,
        target=provider.target,
        num_seeds=num_seeds_to_test,
        opt_level=3
    )

    # Extract processor info safely
    proc = provider.configuration().processor_type
    processor_type = f"{proc['family']}{proc['revision']}"

    # Store the results for the DataFrame
    benchmark_results.append({
        "Backend Name": backend_name,
        "Processor": processor_type,
        "Qubits": provider.configuration().n_qubits,
        "Depth": best_metrics['depth'],
        "2Q Gate Count (CZ/RZZ/ECR)": best_metrics['two_q_count'],
        "Best Seed": best_metrics['seed'],
        "Best Qubits": best_metrics['best_qubits']
    })

    print(f"[{backend_name}] Done! Best Seed: {best_metrics['seed']} | 2Q Gates: {best_metrics['two_q_count']}\n")

# Create and display the Pandas DataFrame
df_best_seeds = pd.DataFrame(benchmark_results)

# Sort by the lowest amount of 2-qubit gates to see the topological winners
df_best_seeds = df_best_seeds.sort_values(by="2Q Gate Count (CZ/RZZ/ECR)", ascending=True).reset_index(drop=True)

display(df_best_seeds)

Starting parallel seed search (500 seeds per backend).
Using 44 CPU cores.

[fake_marrakesh] Scanning seeds...
[fake_marrakesh] Done! Best Seed: 17 | 2Q Gates: 255

[fake_fez] Scanning seeds...
[fake_fez] Done! Best Seed: 14 | 2Q Gates: 255

[fake_kingston] Scanning seeds...
[fake_kingston] Done! Best Seed: 34 | 2Q Gates: 255

[fake_boston] Scanning seeds...
[fake_boston] Done! Best Seed: 14 | 2Q Gates: 255

[fake_pittsburgh] Scanning seeds...
[fake_pittsburgh] Done! Best Seed: 84 | 2Q Gates: 255

[fake_miami] Scanning seeds...
[fake_miami] Done! Best Seed: 177 | 2Q Gates: 240

[fake_aachen] Scanning seeds...
[fake_aachen] Done! Best Seed: 52 | 2Q Gates: 255

[fake_berlin] Scanning seeds...
[fake_berlin] Done! Best Seed: 182 | 2Q Gates: 240



,Backend Name,Processor,Qubits,Depth,2Q Gate Count (CZ/RZZ/ECR),Best Seed,Best Qubits
0,fake_berlin,Nighthawk1,120,746,240,182,"[58, 48, 38, 39, 29, 28]"
1,fake_miami,Nighthawk1,120,748,240,177,"[87, 97, 96, 86, 85, 95]"
2,fake_marrakesh,Heron2,156,763,255,17,"[84, 85, 77, 65, 64, 63]"
3,fake_fez,Heron2,156,765,255,14,"[26, 25, 37, 45, 46, 47]"
4,fake_boston,Heron3,156,765,255,14,"[26, 25, 37, 45, 46, 47]"
5,fake_kingston,Heron2,156,765,255,34,"[47, 48, 49, 50, 51, 58]"
6,fake_pittsburgh,Heron3,156,765,255,84,"[87, 86, 85, 77, 65, 64]"
7,fake_aachen,Heron3,156,765,255,52,"[147, 137, 127, 128, 129, 118]"


In [19]:
backend_best_seeds_df = df_best_seeds.sort_values(by="Backend Name", ascending=True).reset_index(drop=True)
backend_best_seeds_df.to_excel('backend_best_seeds_df.xlsx', index=False)
display(backend_best_seeds_df)

,Backend Name,Processor,Qubits,Depth,2Q Gate Count (CZ/RZZ/ECR),Best Seed,Best Qubits
0,fake_aachen,Heron3,156,765,255,52,"[147, 137, 127, 128, 129, 118]"
1,fake_berlin,Nighthawk1,120,746,240,182,"[58, 48, 38, 39, 29, 28]"
2,fake_boston,Heron3,156,765,255,14,"[26, 25, 37, 45, 46, 47]"
3,fake_fez,Heron2,156,765,255,14,"[26, 25, 37, 45, 46, 47]"
4,fake_kingston,Heron2,156,765,255,34,"[47, 48, 49, 50, 51, 58]"
5,fake_marrakesh,Heron2,156,763,255,17,"[84, 85, 77, 65, 64, 63]"
6,fake_miami,Nighthawk1,120,748,240,177,"[87, 97, 96, 86, 85, 95]"
7,fake_pittsburgh,Heron3,156,765,255,84,"[87, 86, 85, 77, 65, 64]"


# 6°: Mapomatic procedure for detecting the best qubit subgraphs.

In [20]:
import mapomatic as mm

# 1. Helper dictionary to map the backend names in the DataFrame to the actual instantiated objects
provider_dict = {provider.backend_name: provider for provider in fake_providers}

mapomatic_results = []

print("Starting Mapomatic sweep over the optimized topologies...\n")

# 2. Iterate over the DataFrame containing the best seeds for each QPU
for index, row in backend_best_seeds_df.iterrows():
    backend_name = row['Backend Name']
    best_seed = int(row['Best Seed'])
    original_depth = row['Depth']
    original_2q = row['2Q Gate Count (CZ/RZZ/ECR)']

    # Retrieve the actual backend object
    provider = provider_dict[backend_name]

    print(f"[{backend_name}] Transpiling with the Winning Seed ({best_seed})...")

    # 3. Re-transpile the circuit fixing the seed to guarantee the lowest SWAP count
    pm = generate_preset_pass_manager(
        target=provider.target,
        layout_method='sabre',
        routing_method='sabre',
        optimization_level=3,
        seed_transpiler=best_seed
    )

    transpiled_qc = pm.run(ansatz)

    # 4. Deflate: Mapomatic requires a "deflated" circuit
    # This removes inactive physical qubits from the chip layout so the algorithm
    # can accurately match only the active logical subgraph.
    small_qc = mm.deflate_circuit(transpiled_qc)

    print(f"[{backend_name}] Calculating isomorphic subgraphs and evaluating noise...")

    # 5. Find all windows on the chip where this subgraph fits perfectly
    layouts = mm.matching_layouts(small_qc, provider)

    # 6. Evaluate the layouts using Mapomatic's heuristic cost function
    # The result is a list of tuples ordered by the LOWEST estimated error: [(layout, score), ...]
    scores = mm.evaluate_layouts(small_qc, layouts, provider)

    # Extract the absolute winner for this specific machine
    best_layout_data = scores[0]
    best_qubits = best_layout_data[0]
    best_error_score = best_layout_data[1]

    # Store the data
    mapomatic_results.append({
        "Backend Name": backend_name,
        "Processor": row['Processor'],
        "Seed Used": best_seed,
        "Depth": original_depth,
        "2Q Gates": original_2q,
        "Mapomatic Best Qubits": best_qubits,
        "Mapomatic Error Score": best_error_score
    })

    # Alterado para notação padrão com 6 casas decimais (.6f)
    print(f"[{backend_name}] Done! Best error score: {best_error_score:.6f}\n")

# =====================================================================
# Final Table Generation (The Grand Ranking)
# =====================================================================

df_mapomatic_final = pd.DataFrame(mapomatic_results)

# Mapomatic's golden metric is the Error Score (lower means higher fidelity).
# We sort from lowest error to highest.
df_mapomatic_final = df_mapomatic_final.sort_values(by="Mapomatic Error Score", ascending=True).reset_index(drop=True)

# Aesthetic formatting for the final report (using standard float notation)
df_mapomatic_styled = df_mapomatic_final.style.format({
    "Mapomatic Error Score": "{:.6f}"
})

display(df_mapomatic_styled)

Starting Mapomatic sweep over the optimized topologies...

[fake_aachen] Transpiling with the Winning Seed (52)...
[fake_aachen] Calculating isomorphic subgraphs and evaluating noise...
[fake_aachen] Done! Best error score: 0.268511

[fake_berlin] Transpiling with the Winning Seed (182)...
[fake_berlin] Calculating isomorphic subgraphs and evaluating noise...
[fake_berlin] Done! Best error score: 0.276551

[fake_boston] Transpiling with the Winning Seed (14)...
[fake_boston] Calculating isomorphic subgraphs and evaluating noise...
[fake_boston] Done! Best error score: 0.252457

[fake_fez] Transpiling with the Winning Seed (14)...
[fake_fez] Calculating isomorphic subgraphs and evaluating noise...
[fake_fez] Done! Best error score: 0.533403

[fake_kingston] Transpiling with the Winning Seed (34)...
[fake_kingston] Calculating isomorphic subgraphs and evaluating noise...
[fake_kingston] Done! Best error score: 0.301407

[fake_marrakesh] Transpiling with the Winning Seed (17)...
[fake_mar

,Backend Name,Processor,Seed Used,Depth,2Q Gates,Mapomatic Best Qubits,Mapomatic Error Score
0,fake_boston,Heron3,14,765,255,"[43, 42, 56, 63, 62, 61]",0.252457
1,fake_aachen,Heron3,52,765,255,"[153, 139, 155, 154, 135, 134]",0.268511
2,fake_berlin,Nighthawk1,182,746,240,"[10, 20, 11, 21, 12, 22]",0.276551
3,fake_pittsburgh,Heron3,84,765,255,"[97, 87, 86, 85, 84, 83]",0.284731
4,fake_kingston,Heron2,34,765,255,"[23, 16, 3, 2, 1, 0]",0.301407
5,fake_marrakesh,Heron2,17,763,255,"[19, 15, 14, 13, 11, 12]",0.307387
6,fake_miami,Nighthawk1,177,748,240,"[11, 10, 40, 21, 20, 30]",0.353848
7,fake_fez,Heron2,14,765,255,"[25, 26, 37, 45, 44, 43]",0.533403


In [21]:
df_mapomatic_final = df_mapomatic_final.sort_values(by="Backend Name", ascending=True).reset_index(drop=True)
display(df_mapomatic_final)

,Backend Name,Processor,Seed Used,Depth,2Q Gates,Mapomatic Best Qubits,Mapomatic Error Score
0,fake_aachen,Heron3,52,765,255,"[153, 139, 155, 154, 135, 134]",0.268511
1,fake_berlin,Nighthawk1,182,746,240,"[10, 20, 11, 21, 12, 22]",0.276551
2,fake_boston,Heron3,14,765,255,"[43, 42, 56, 63, 62, 61]",0.252457
3,fake_fez,Heron2,14,765,255,"[25, 26, 37, 45, 44, 43]",0.533403
4,fake_kingston,Heron2,34,765,255,"[23, 16, 3, 2, 1, 0]",0.301407
5,fake_marrakesh,Heron2,17,763,255,"[19, 15, 14, 13, 11, 12]",0.307387
6,fake_miami,Nighthawk1,177,748,240,"[11, 10, 40, 21, 20, 30]",0.353848
7,fake_pittsburgh,Heron3,84,765,255,"[97, 87, 86, 85, 84, 83]",0.284731


In [22]:
df_mapomatic_final.to_excel('mapomatic_global_ranking.xlsx', index=False)